# BERTopic: Topic Continuity Rate

**Scenario 3b** — Best-match assignment with mismatch detection.

| Priority | Category | Condition |
|----------|----------|-----------|
| 1 | **Merge** | sim > THRESHOLD, and the t+1-topic has >1 source |
| 2 | **Disappear** | sim == 0 |
| 3 | **Evolve** | 0 < sim ≤ THRESHOLD |
| 4 | **Mismatch** | sim > THRESHOLD, single source, best-match ID ≠ own topic ID |
| 5 | **Stable** | sim > THRESHOLD, best-match ID = own topic ID |
| — | **New** | t+1-topic has no incoming source with sim > 0 |

From t-side: Disappear + Merge + Mismatch + Stable + Evolve = 100%


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
TEMPORAL_DIR = Path("../../../../results/bertopic/temporal")
RESULT_DIR = Path("../../../../results/bertopic/continuity")
THRESHOLD = 0.5

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Threshold Disappear : sim == 0")
print(f"Threshold Evolve    : 0 < sim <= {THRESHOLD}")
print(f"Threshold Stable    : sim > {THRESHOLD}")
print(f"Merge requires all sources to have sim > {THRESHOLD}")
print(f"New topic : no incoming source with sim > 0")
print(f"Reading from: {TEMPORAL_DIR}")
print(f"Saving to: {RESULT_DIR}")


Threshold Disappear : sim == 0
Threshold Evolve    : 0 < sim <= 0.5
Threshold Stable    : sim > 0.5
Merge requires all sources to have sim > 0.5
New topic : no incoming source with sim > 0
Reading from: ../../../../results/bertopic/temporal
Saving to: ../../../../results/bertopic/continuity


In [3]:
def parse_words(words_str):
    return [w.strip() for w in str(words_str).split(",")]


def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

## Compute Continuity Rate (Best-Match)

In [4]:
for subject in LIST_SUBJECT:
    print(f"\n'======================================================================'")
    print(f"Continuity Rate: {subject.upper()} (BERTopic)")
    print(f"'======================================================================'")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        topic_words[key] = parse_words(row["top_words"])

    years = sorted(evo_df["year"].unique())

    all_transition_rows = []
    all_merge_rows = []
    all_new_rows = []
    summary_rows = []

    for i in range(len(years) - 1):
        t, t_next = int(years[i]), int(years[i + 1])

        topics_t  = sorted([tid for (y, tid) in topic_words if y == t])
        topics_t1 = sorted([tid for (y, tid) in topic_words if y == t_next])

        if not topics_t or not topics_t1:
            continue

        # Build full RBO similarity matrix (t → t+1)
        sim_matrix = np.zeros((len(topics_t), len(topics_t1)))
        for ii, tid_t in enumerate(topics_t):
            words_t = topic_words.get((t, tid_t), [])
            for jj, tid_t1 in enumerate(topics_t1):
                words_t1_tmp = topic_words.get((t_next, tid_t1), [])
                sim_matrix[ii, jj] = rbo(words_t, words_t1_tmp, p=0.9)

        # ── Step 1: Each t-topic → its best t+1-topic ────────────────────────
        best_match_idx = np.argmax(sim_matrix, axis=1)
        best_match_sim = np.max(sim_matrix, axis=1)

        # ── Step 2: Count sources (sim > 0.2) per t+1-topic ──────────────────
        target_counts = Counter()
        for idx in range(len(topics_t)):
            if float(best_match_sim[idx]) > THRESHOLD:
                target_counts[topics_t1[best_match_idx[idx]]] += 1

        # merge_targets: t+1-topics claimed by >1 source with sim > 0.2
        merge_targets = {tgt for tgt, cnt in target_counts.items() if cnt > 1}

        # ── Step 3: Classify each t-topic ─────────────────────────────────────
        n_stable = n_evolve = n_merge = n_disappear = n_mismatch = 0

        for idx, tid in enumerate(topics_t):
            sim_val    = float(best_match_sim[idx])
            target_tid = topics_t1[best_match_idx[idx]]
            words_t    = ", ".join(topic_words.get((t, tid), []))

            # ── 1. MERGE: target has >1 source ───────────────────────────────
            if tid in merge_targets:
                category = "merge"
                n_merge += 1

            # ── 2. DISAPPEAR: sim == 0 ────────────────────────────────────────
            elif sim_val == 0:
                category = "disappear"
                n_disappear += 1

            # ── 3. EVOLVE: 0 < sim <= 0.5 ─────────────────────────────────────
            elif sim_val > 0 and sim_val <= THRESHOLD:
                category = "evolve"
                n_evolve += 1

            # ── 4. MISMATCH: sim > 0.5, single source, target ID ≠ own ID ────
            elif target_tid != tid:
                category   = "mismatch"
                n_mismatch += 1

            # ── 5. STABLE: sim > 0.5, target ID = own ID ─────────────────────
            else:
                category = "stable"
                n_stable += 1

            all_transition_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "topic_id": tid, "category": category,
                "best_match_topic": target_tid,
                "best_match_sim": round(sim_val, 6),
                "words": words_t,
            })

        # ── Step 4: Record merge groups ────────────────────────────────────────
        for target_tid in merge_targets:
            sources = [tid for idx, tid in enumerate(topics_t)
                       if topics_t1[best_match_idx[idx]] == target_tid
                       and float(best_match_sim[idx]) > THRESHOLD]
            source_sims = [round(float(sim_matrix[topics_t.index(s),
                                                   topics_t1.index(target_tid)]), 4)
                           for s in sources]
            words_t1 = ", ".join(topic_words.get((t_next, target_tid), []))
            all_merge_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "target_topic": target_tid,
                "n_sources": len(sources),
                "source_topics": str(sources),
                "source_sims": str(source_sims),
                "target_words": words_t1,
            })

        # ── Step 5: New topics — no incoming source with sim > 0 ────────────
        new_topics = []
        for jj, tid in enumerate(topics_t1):
            if np.max(sim_matrix[:, jj]) == 0:
                new_topics.append(tid)
        for tid in new_topics:
            words_t1 = ", ".join(topic_words.get((t_next, tid), []))
            all_new_rows.append({
                "subject": subject, "year_from": t, "year_to": t_next,
                "topic_id": tid, "words": words_t1,
            })

        total_t        = len(topics_t)
        n_new          = len(new_topics)
        n_merge_groups = len(merge_targets)

        summary_rows.append({
            "subject": subject, "year_from": t, "year_to": t_next,
            "n_topics_t": total_t, "n_topics_t1": len(topics_t1),
            "n_stable": n_stable, "n_evolve": n_evolve, "n_merge": n_merge,
            "n_mismatch": n_mismatch, "n_disappear": n_disappear,
            "n_merge_groups": n_merge_groups, "n_new": n_new,
            "pct_stable":    round(n_stable    / total_t * 100, 2),
            "pct_evolve":    round(n_evolve    / total_t * 100, 2),
            "pct_merge":     round(n_merge     / total_t * 100, 2),
            "pct_mismatch":  round(n_mismatch  / total_t * 100, 2),
            "pct_disappear": round(n_disappear / total_t * 100, 2),
        })

        print(f"  {t}→{t_next}: Stable={n_stable} ({n_stable/total_t:.0%})  "
              f"Evolve={n_evolve} ({n_evolve/total_t:.0%})  "
              f"Merge={n_merge} ({n_merge/total_t:.0%})  "
              f"Mismatch={n_mismatch} ({n_mismatch/total_t:.0%})  "
              f"Disappear={n_disappear} ({n_disappear/total_t:.0%})  "
              f"New={n_new}")

    # Save CSVs
    pd.DataFrame(all_transition_rows).to_csv(
        RESULT_DIR / subject / "continuity_transitions.csv", index=False)
    pd.DataFrame(all_merge_rows).to_csv(
        RESULT_DIR / subject / "continuity_merges.csv", index=False)
    pd.DataFrame(all_new_rows).to_csv(
        RESULT_DIR / subject / "continuity_new_topics.csv", index=False)

    sum_df = pd.DataFrame(summary_rows)
    sum_df.to_csv(RESULT_DIR / subject / "continuity_summary.csv", index=False)

    subj_sum = sum_df[sum_df["subject"] == subject]
    overall = {
        "subject": subject,
        "threshold_stable":    THRESHOLD,
        "avg_pct_stable":    round(subj_sum["pct_stable"].mean(),    2),
        "avg_pct_evolve":    round(subj_sum["pct_evolve"].mean(),    2),
        "avg_pct_merge":     round(subj_sum["pct_merge"].mean(),     2),
        "avg_pct_mismatch":  round(subj_sum["pct_mismatch"].mean(),  2),
        "avg_pct_disappear": round(subj_sum["pct_disappear"].mean(), 2),
        "total_merge_groups": int(subj_sum["n_merge_groups"].sum()),
        "total_new":          int(subj_sum["n_new"].sum()),
    }
    pd.DataFrame([overall]).to_csv(
        RESULT_DIR / subject / "continuity_overall.csv", index=False)

    print(f"\n  Overall: Stable={overall['avg_pct_stable']:.1f}%  "
          f"Evolve={overall['avg_pct_evolve']:.1f}%  "
          f"Merge={overall['avg_pct_merge']:.1f}%  "
          f"Mismatch={overall['avg_pct_mismatch']:.1f}%  "
          f"Disappear={overall['avg_pct_disappear']:.1f}%  "
          f"New={overall['total_new']}")
    print(f"  Saved: {RESULT_DIR / subject}")



'======================================================================'
Continuity Rate: CS (BERTopic)
'======================================================================'
  2000→2001: Stable=2 (4%)  Evolve=39 (83%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=6 (13%)  New=19
  2001→2002: Stable=3 (5%)  Evolve=43 (68%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=17 (27%)  New=21
  2002→2003: Stable=2 (3%)  Evolve=50 (77%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=13 (20%)  New=22
  2003→2004: Stable=1 (1%)  Evolve=60 (81%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=13 (18%)  New=25
  2004→2005: Stable=0 (0%)  Evolve=72 (76%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=23 (24%)  New=21
  2005→2006: Stable=2 (2%)  Evolve=73 (84%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=12 (14%)  New=19
  2006→2007: Stable=4 (5%)  Evolve=62 (70%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=22 (25%)  New=16
  2007→2008: Stable=4 (5%)  Evolve=67 (82%)  Merge=0 (0%)  Mismatch=0 (0%)  Disappear=11 (13%)  New=18

## Analysis Review

In [5]:
LIST_SUBJECT = ["cs", "math", "physics"]
TEMPORAL_DIR = Path("../../../../results/bertopic/temporal")
RESULT_DIR = Path("../../../../results/bertopic/continuity")
THRESHOLD = 0.5

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Threshold Disappear : sim == 0")
print(f"Threshold Evolve    : 0 < sim <= {THRESHOLD}")
print(f"Threshold Stable    : sim > {THRESHOLD}")
print(f"Merge requires all sources to have sim > {THRESHOLD}")
print(f"New topic : no incoming source with sim > 0")
print(f"Reading from: {TEMPORAL_DIR}")
print(f"Saving to: {RESULT_DIR}")


Threshold Disappear : sim == 0
Threshold Evolve    : 0 < sim <= 0.5
Threshold Stable    : sim > 0.5
Merge requires all sources to have sim > 0.5
New topic : no incoming source with sim > 0
Reading from: ../../../../results/bertopic/temporal
Saving to: ../../../../results/bertopic/continuity
